# CineMatch: Movie Recommendation and Rating Analysis System

CineMatch loads real movie and rating data from the MovieLens dataset and uses it to recommend films based on a user's watch history and genre preferences. It also produces visualizations of rating distributions and genre trends. This notebook runs the full system from data loading through recommendations and charts.

In [ ]:
import sys
import os

# make sure the modules in this directory are importable
sys.path.insert(0, os.getcwd())

from cinematch.movie import Movie
from cinematch.user import User
from cinematch.recommendation_system import RecommendationSystem
from cinematch.utils import (
    get_all_genres,
    generate_movie_report,
    random_movie_suggestion,
    genre_filter_generator,
    calculate_popularity_scores
)

print('All modules imported successfully.')

In [ ]:
# create the system and load both data files
system = RecommendationSystem()
system.load_movies('data/movies.csv')
system.load_ratings('data/ratings.csv')

print(str(system))
print(f'Total movies loaded: {len(system)}')

In [ ]:
# add 3 users with different tastes
system.add_user(1, 'Alice')
system.add_user(2, 'Bob')
system.add_user(3, 'Carol')

system.users[1].set_preferred_genres(['Action', 'Sci-Fi'])
system.users[2].set_preferred_genres(['Drama', 'Romance'])
system.users[3].set_preferred_genres(['Comedy', 'Animation'])

# add some movies from the MovieLens dataset to each user's history
for movie_id in [1, 2, 3, 32, 110]:
    system.users[1].add_to_history(movie_id)

for movie_id in [47, 50, 110, 150]:
    system.users[2].add_to_history(movie_id)

for movie_id in [1, 10, 150, 260]:
    system.users[3].add_to_history(movie_id)

for user in system.users.values():
    print(user)

In [ ]:
# compute and print average rating per genre
genre_stats = system.compute_genre_stats()
print('Average rating by genre:')
for genre, avg in genre_stats.items():
    print(f'  {genre}: {avg}')

print()
print('Top 5 rated movies:')
top_movies = system.get_top_movies(5)
for i, movie in enumerate(top_movies, 1):
    print(f'  {i}. {str(movie)}')

In [ ]:
# recommendations for Alice (user 1)
print('Recommendations for User 1 (Alice):')
recs = system.recommend_for_user(1, n=5)
for movie in recs:
    print(f'  {movie}')

print()
print('First 5 Action movies (using generator):')
count = 0
for movie in genre_filter_generator(system.movies, 'Action'):
    print(f'  {movie}')
    count += 1
    if count >= 5:
        break

print()
suggestion = random_movie_suggestion(system.movies)
print(f'Random movie suggestion: {suggestion}')

In [ ]:
# generate and save both charts
system.plot_genre_ratings()
system.plot_rating_distribution()

In [ ]:
# print all unique genres found in the dataset
all_genres = get_all_genres(system.movies)
print(f'All genres in the dataset ({len(all_genres)} total):')
print(all_genres)

print()
print('Top 5 movies by popularity score:')
pop_scores = calculate_popularity_scores(system.movies)
for title, score in pop_scores[:5]:
    print(f'  {title}: {score:.2f}')

print()
print('Movie report for top 5 rated movies:')
top5 = system.get_top_movies(5)
report = generate_movie_report(top5)
for entry in report:
    print(f'  {entry}')

In [ ]:
# pick two top movies and show their similarity score
top10 = system.get_top_movies(10)
movie_a = top10[0]
movie_b = top10[4]

try:
    sim = system.similarity_score(movie_a, movie_b)
    print(f'Similarity score between:')
    print(f'  A: {movie_a.title}')
    print(f'  B: {movie_b.title}')
    print(f'  Score: {sim}')
except RuntimeError as e:
    print(f'Could not compute similarity: {e}')

print()
# set operations — compare which genres each user has watched
user1_genres = {system.movies[mid].genre for mid in system.users[1].watch_history if mid in system.movies}
user2_genres = {system.movies[mid].genre for mid in system.users[2].watch_history if mid in system.movies}

print(f'User 1 genres: {user1_genres}')
print(f'User 2 genres: {user2_genres}')
print(f'Union (all genres between both users): {user1_genres | user2_genres}')
print(f'Intersection (genres both users watched): {user1_genres & user2_genres}')
print(f'Difference (User 1 genres not in User 2): {user1_genres - user2_genres}')

## Summary

CineMatch loads and processes approximately 9,000 movies and 100,000 ratings from the MovieLens dataset to build a working recommendation system. The system uses object-oriented design with composition — the `RecommendationSystem` class holds `Movie` and `User` objects — and pandas for data loading and genre aggregation, and matplotlib for bar charts and histograms.

Through this project we learned how to break a larger Python program across multiple files and classes, how to use pandas `groupby` and `mean` alongside pure Python data structures, and how functional tools like `map`, `filter`, `lambda`, and generators make data processing more concise. The recommendation and similarity scoring logic showed us how to think about ranking and comparing items without any machine learning libraries — just math and sorted lists.